# ST-OMR Meter V5-3C guarded secondary witness audit

One exact-SHA diagnostic-only run. No candidate checkpoint, model mutation, retention, or validation.

In [ ]:
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import time

EXPECTED_HEAD = "61361612abfce132994abaca742c855f91305b44"
EXPECTED_CI_RUN_ID = 32751453021
EXPECTED_SCIPY_VERSION = "1.18.0"
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO = Path("/content/st-omr-training")
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = MYDRIVE / "ST-OMR-D10" / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
for name, path in {"DATA_ROOT": DATA_ROOT, "CHECKPOINT_ROOT": CHECKPOINT_ROOT, "M4A_ROOT": M4A_ROOT, "D10_ROOT": D10_ROOT}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} bulunamadi: {path}")
print("DRIVE CHECK = PASS")

if not REPO.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"REPO git repository degil: {REPO}")
remotes = subprocess.check_output(["git", "-C", str(REPO), "remote"], text=True).split()
if "origin" not in remotes:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "add", "origin", REPO_URL])
else:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL])
subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", EXPECTED_HEAD, "--depth", "1"])
fetched_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True).strip()
if fetched_head != EXPECTED_HEAD:
    raise RuntimeError(f"FETCH_HEAD mismatch: expected={EXPECTED_HEAD} actual={fetched_head}")
subprocess.check_call(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD])
actual_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual_head != EXPECTED_HEAD:
    raise RuntimeError(f"HEAD mismatch: {actual_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository worktree temiz degil")
print("REPOSITORY CHECK = PASS")
print("HEAD =", actual_head)
print("CI RUN ID =", EXPECTED_CI_RUN_ID)

if metadata.version("scipy") != EXPECTED_SCIPY_VERSION:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "scipy==1.18.0"])
if metadata.version("scipy") != EXPECTED_SCIPY_VERSION:
    raise RuntimeError(f"SciPy version mismatch: {metadata.version('scipy')}")
print("SCIPY VERSION CHECK = PASS", metadata.version("scipy"))

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from st_omr_training import meter_v5_1_bbox_pilot as v51
from st_omr_training import meter_v5_2b_specialist_adaptation as v52b
from st_omr_training import meter_v5_3a_robust_margin_head_candidate_v1 as v53a
from st_omr_training import meter_v5_3c_guarded_secondary_witness_audit_v1 as audit
print("MODULE IMPORT = PASS")
if audit.APPROVAL_TOKEN != "V5_3C_SINGLE_GUARDED_SECONDARY_AUDIT_APPROVED":
    raise RuntimeError("Approval token contract changed")

DIGIT2_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT2_SHA256)
DIGIT3_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT3_SHA256)
ANN_DIR = DATA_ROOT / "annotations"
V53A_REPORT = ANN_DIR / v53a.REPORT_NAME
V53A_RECOVERY = ANN_DIR / f"v5_3a_render_recovery_envelope_{audit.V53A_SOURCE_HARNESS_HEAD}.json"
REPORT_PATH = ANN_DIR / audit.REPORT_NAME
ENVELOPE_PATH = ANN_DIR / f"v5_3c_execution_envelope_{EXPECTED_HEAD}.json"
for name, path in {"V5-3A REPORT": V53A_REPORT, "V5-3A RECOVERY": V53A_RECOVERY}.items():
    if not path.is_file():
        raise RuntimeError(f"{name} missing: {path}")
for path in (REPORT_PATH, ENVELOPE_PATH):
    if path.exists():
        raise RuntimeError(f"Refusing overwrite/rerun: {path}")
print("EXACT INPUT BINDING = PASS")
print("OUTPUT GUARD = PASS")

required_safety = {
    "training": False,
    "secondary_linear_program_witness_fit": True,
    "primary_linear_program_rerun": False,
    "candidate_checkpoint_write_authorized": False,
    "candidate_checkpoint_written": False,
    "model_parameter_mutation_executed": False,
    "autograd_grad_used": False,
    "backward": False,
    "optimizer_steps": 0,
    "frozen_backbone": True,
    "frozen_head_bias": True,
    "runtime_threshold_tuning": False,
    "alternative_threshold_evaluated": False,
    "solver_sweep": False,
    "fallback_solver": False,
    "new_bbox": False,
    "new_crop_geometry": False,
    "new_spatial_heuristic": False,
    "historical_retention_executed": False,
    "historical_validation_opened": False,
    "first30_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
    "digit4_frozen": True,
    "production_promotion": False,
}
for key, expected in required_safety.items():
    if audit.safety_boundary().get(key) != expected:
        raise RuntimeError(f"Safety boundary mismatch: {key}")
contract = audit.solver_contract()
for key, expected in {
    "library_version_matches_expected": True,
    "witness_tolerance": 1e-7,
    "primary_l1_absolute_slack": 1e-6,
    "internal_cap_guard": 5e-7,
    "secondary_l1_cap_row_normalized_to_rhs_one": True,
    "external_acceptance_cap_unchanged": True,
    "primary_lp_rerun": False,
    "solver_sweep": False,
    "fallback_solver": False,
    "tolerance_changed": False,
    "objective_changed": False,
    "margin_changed": False,
    "threshold_or_bias_changed": False,
}.items():
    if contract.get(key) != expected:
        raise RuntimeError(f"Solver contract mismatch: {key}")
print("V5-3C CONTRACT = PASS")
print("TOLERANCE UNCHANGED | EXTERNAL CAP UNCHANGED")
print("CANDIDATE WRITE = FORBIDDEN | RETENTION = CLOSED")

started = time.time()
def progress(processed, total, phase):
    if processed == 1 or processed == total or processed % 2048 == 0:
        print(phase, f"{processed}/{total}", f"| elapsed={int(time.time() - started)}s")

report = audit.run_guarded_secondary_witness_audit_v1(
    DATA_ROOT,
    m4a_root=M4A_ROOT,
    d10_root=D10_ROOT,
    digit2_frozen=DIGIT2_FROZEN,
    digit3_frozen=DIGIT3_FROZEN,
    v5_3a_report=V53A_REPORT,
    v5_3a_recovery_envelope=V53A_RECOVERY,
    confirmation=audit.APPROVAL_TOKEN,
    progress=progress,
)
if not REPORT_PATH.is_file():
    raise RuntimeError(f"Audit report missing: {REPORT_PATH}")
report_bytes = REPORT_PATH.read_bytes()
saved_report = json.loads(report_bytes.decode("utf-8"))
if saved_report != report:
    raise RuntimeError("Saved report mismatch")
for key, expected in required_safety.items():
    if report.get(key) != expected:
        raise RuntimeError(f"Saved report safety mismatch: {key}")
if report.get("witness_weight_values_emitted") is not False:
    raise RuntimeError("Witness weight values were emitted")
gate = report.get("diagnostic_witness_gate")
if gate not in {"PASS", "HOLD"}:
    raise RuntimeError(f"Unknown diagnostic gate: {gate}")
for digit in ("2", "3"):
    specialist = report["per_specialist"][digit]
    result = specialist["guarded_secondary"]
    if result.get("witness_claim") == "GUARDED_SECONDARY_WITNESS_VERIFIED":
        for key in ("external_l1_cap_violations", "internal_guarded_l1_cap_violations", "primary_lower_bound_conflicts", "parameter_bound_violations", "v5_constraint_violations", "v5_solver_margin_constraint_violations", "historical_margin_constraint_violations", "historical_solver_margin_constraint_violations"):
            if result.get(key) != 0:
                raise RuntimeError(f"{digit}-AI residual violation: {key}")
        if result.get("functional_delta_identity_verified") is not True:
            raise RuntimeError(f"{digit}-AI functional identity failed")
        if result["diagnostic_v5_train_metrics"].get("f1") != 1.0:
            raise RuntimeError(f"{digit}-AI V5 TRAIN F1 changed")
        if result["historical_transition_counts"].get("correct_to_wrong") != 0:
            raise RuntimeError(f"{digit}-AI historical frozen-correct regression")
        if specialist.get("float32_copy_gate", {}).get("gate") != "PASS":
            raise RuntimeError(f"{digit}-AI float32 copy gate HOLD")

post_run_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if post_run_head != EXPECTED_HEAD:
    raise RuntimeError(f"Post-run HEAD mismatch: {post_run_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository changed during audit")
report_sha256 = hashlib.sha256(report_bytes).hexdigest()
envelope = {
    "schema": "st-omr-meter-v5-3c-exact-sha-execution-envelope-v1",
    "repository": REPOSITORY,
    "expected_head": EXPECTED_HEAD,
    "actual_head_before_run": actual_head,
    "actual_head_after_run": post_run_head,
    "ci_run_id": EXPECTED_CI_RUN_ID,
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "report_sha256": report_sha256,
    "diagnostic_witness_gate": gate,
    "claims": {digit: report["per_specialist"][digit]["guarded_secondary"]["witness_claim"] for digit in ("2", "3")},
    "candidate_checkpoint_written": False,
    "model_parameter_mutation_executed": False,
    "safety_boundary": {key: report[key] for key in required_safety},
}
v51._atomic_write_json(ENVELOPE_PATH, envelope)
envelope_sha256 = hashlib.sha256(ENVELOPE_PATH.read_bytes()).hexdigest()

print()
print("============================================")
print("V5-3C GUARDED SECONDARY WITNESS RESULT")
print("============================================")
print("DIAGNOSTIC WITNESS GATE =", gate)
for digit in ("2", "3"):
    specialist = report["per_specialist"][digit]
    result = specialist["guarded_secondary"]
    print()
    print(f"========== {digit}-AI ==========")
    print("CLAIM =", result.get("witness_claim"))
    print("SOLVER =", {key: result.get(key) for key in ("status", "success", "iterations", "optimality_claim")})
    print("PRIMARY L1 =", result.get("exact_primary_l1_optimum"))
    print("EXTERNAL CAP =", result.get("external_l1_acceptance_cap"))
    print("INTERNAL GUARDED CAP =", result.get("internal_guarded_l1_cap"))
    print("RECOMPUTED L1 =", result.get("independently_recomputed_delta_weight_l1"))
    print("EXTERNAL CAP EXCESS =", result.get("external_l1_cap_excess"))
    print("INTERNAL CAP EXCESS =", result.get("internal_guarded_l1_cap_excess"))
    print("RESIDUALS =", {key: result.get(key) for key in ("external_l1_cap_violations", "internal_guarded_l1_cap_violations", "primary_lower_bound_conflicts", "parameter_bound_violations", "v5_solver_margin_constraint_violations", "historical_solver_margin_constraint_violations")})
    print("V5 TRAIN METRICS =", result.get("diagnostic_v5_train_metrics"))
    print("HISTORICAL TRAIN METRICS =", result.get("diagnostic_historical_train_metrics"))
    print("HISTORICAL TRANSITIONS =", result.get("historical_transition_counts"))
    print("WEIGHT GEOMETRY =", result.get("weight_geometry"))
    print("FLOAT32 COPY GATE =", specialist.get("float32_copy_gate", "NOT_RUN_DUE_TO_HOLD"))
print()
print("EXACT SHA EXECUTION = PASS")
print("HEAD =", post_run_head)
print("REPORT =", REPORT_PATH)
print("REPORT SHA256 =", report_sha256)
print("EXECUTION ENVELOPE =", ENVELOPE_PATH)
print("ENVELOPE SHA256 =", envelope_sha256)
print("TRAINING EXECUTED = False")
print("CANDIDATE CHECKPOINT WRITTEN = False")
print("MODEL PARAMETER MUTATION = False")
print("HISTORICAL RETENTION = NOT RUN")
print("FIRST-30 = CLOSED | V5 VAL = CLOSED | FINAL HOLDOUT = LOCKED")
